In [1]:
# %% [markdown]
# # 02_coverage_analysis.ipynb
# Static analysis of the 60-question credit rating benchmark against
# the current Semantic Layer YAML's concept vocabulary. No API calls,
# no database connection required -- this only reads
# credit_rating_questions_all.json and financial_semantic_layer.yaml.
#
# Two checks:
#   1. CONCEPT COVERAGE: does every raw column referenced in each
#      gold SQL query have a corresponding concept (dimension or
#      measure) defined in the YAML? A gap here means the v2
#      architecture literally cannot express that gold query's logic
#      through concept names alone.
#   2. MEASURE-MISUSE RISK: which questions have a WHERE-clause shape
#      that could tempt an LLM into using an aggregate MEASURE as a
#      row-level filter condition -- the exact failure pattern found
#      in the pilot run (CR01: "WHERE loan.default_count > 0" instead
#      of "WHERE loan.status = 'D'"), which the compiler's WHERE-guard
#      now catches as a compile_error rather than invalid SQL, but
#      which still costs API budget and lowers effective accuracy if
#      it happens often.
#
# Run this BEFORE any pilot/full run so YAML gaps are fixed and
# few-shot examples are targeted at the highest-risk patterns first,
# rather than discovering them one API call at a time.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import json
import re
import yaml
from collections import Counter

# %% [markdown]
# ## 1. Load the question set and the Semantic Layer YAML

# %%
QUESTIONS_PATH = "./credit_rating_questions_all.json"
YAML_PATH = "./financial_semantic_layer.yaml"

with open(QUESTIONS_PATH, "r", encoding="utf-8") as f:
    questions = json.load(f)

with open(YAML_PATH, "r", encoding="utf-8") as f:
    sl = yaml.safe_load(f)

print(f"Loaded {len(questions)} questions from {QUESTIONS_PATH}")
print(f"Loaded {len(sl['cubes'])} cubes from {YAML_PATH}")

# Sanity check on question fields -- adapt downstream code if the
# actual file uses different field names than expected.
if len(questions) > 0:
    print(f"\nFields present in each question record: {sorted(questions[0].keys())}")
    required_fields = {'question_id', 'difficulty', 'category', 'SQL'}
    missing = required_fields - set(questions[0].keys())
    if missing:
        print(f"\n!! WARNING: expected fields not found: {missing}")
        print("   Downstream cells assume these field names -- check the")
        print("   actual JSON structure and adjust if it differs.")
    else:
        print("✓ All expected fields present.")

if len(questions) != 60:
    print(f"\n!! WARNING: expected 60 questions, found {len(questions)}.")
    print("   credit_rating_questions_all.json may be a partial merge or")
    print("   a different version than the one used in the manuscript.")

# %% [markdown]
# ## 2. Build raw-column -> concept reverse lookup from the YAML
#     (mirrors semantic_compiler.py's own dimension/measure parsing
#     logic, kept independent here deliberately -- this notebook is
#     meant to be a check ON the compiler's config, not a reuse of
#     the compiler's internals, so a bug in one won't be masked by
#     reusing the same code path in the other)

# %%
raw_to_concept = {}   # table -> {raw_col: concept_name}  (simple dims only)
known_tables = set()

for cube in sl['cubes']:
    table = cube['sql_table']
    known_tables.add(table)
    raw_to_concept.setdefault(table, {})
    for d in cube.get('dimensions', []):
        sql_val = d['sql'].strip() if isinstance(d['sql'], str) else str(d['sql'])
        is_bare = sql_val.replace('_', '').isalnum() and ' ' not in sql_val and '(' not in sql_val
        if is_bare:
            raw_to_concept[table][sql_val] = d['name']

print(f"Known tables: {sorted(known_tables)}")
print(f"\nSimple dimension raw-column mappings per table:")
for table, mapping in raw_to_concept.items():
    if mapping:
        print(f"  {table}: {mapping}")

# %% [markdown]
# ## 3. Known physical columns per table (for detecting genuinely
#     uncovered columns vs. query-local aliases). Mirrors utils.py's
#     FINANCIAL_TABLES constant -- kept in sync manually since this
#     notebook intentionally doesn't import utils.py (to stay usable
#     even before utils.py/the DB connection is fully working).

# %%
FINANCIAL_TABLES = {
    "account" : ["account_id", "district_id", "frequency", "date"],
    "client"  : ["client_id", "gender", "birth_date", "district_id"],
    "disp"    : ["disp_id", "client_id", "account_id", "type"],
    "loan"    : ["loan_id", "account_id", "date", "amount", "duration", "payments", "status"],
    "trans"   : ["trans_id", "account_id", "date", "type", "operation",
                 "amount", "balance", "k_symbol", "bank", "account"],
    "card"    : ["card_id", "disp_id", "type", "issued"],
    "order"   : ["order_id", "account_id", "bank_to", "account_to", "amount", "k_symbol"],
    "district": ["district_id", "A2", "A3", "A4", "A5", "A6", "A7", "A8",
                 "A9", "A10", "A11", "A12", "A13", "A14", "A15", "A16"],
}

JOIN_KEY_COLUMNS = {
    ('account', 'district_id'),
    ('disp', 'account_id'), ('disp', 'client_id'),
    ('card', 'disp_id'),
    ('trans', 'account_id'),
    ('loan', 'account_id'),
}

def find_covering_concept(table, raw_col):
    """
    Returns (concept_name_or_marker, kind) where kind is one of:
      'dimension'              -- covered by a simple dimension
      'measure'                -- referenced inside a measure's SQL/filters
      'join_key'                -- a foreign key, resolved by the compiler's
                                    separate join-hint mechanism, not by
                                    concept substitution
      'uncovered_known_column' -- a real physical column with NO concept
      'unknown'                -- not a known physical column at all
                                    (likely a query-local alias)
    """
    if (table, raw_col) in JOIN_KEY_COLUMNS:
        return "[join key -- resolved via join-hint, not a concept gap]", 'join_key'

    if table in raw_to_concept and raw_col in raw_to_concept[table]:
        return raw_to_concept[table][raw_col], 'dimension'

    for cube in sl['cubes']:
        if cube['sql_table'] != table:
            continue
        for m in cube.get('measures', []):
            text = m['sql']
            for filt in m.get('filters', []):
                text += ' ' + filt['sql']
            if re.search(r'\b' + re.escape(raw_col) + r'\b', text):
                return m['name'], 'measure'

    if table in FINANCIAL_TABLES and raw_col in FINANCIAL_TABLES[table]:
        return None, 'uncovered_known_column'

    return None, 'unknown'

print("Reverse lookup and coverage-check function ready.")

# %% [markdown]
# ## 4. Check every question's SQL for table.column references with
#     no concept coverage

# %%
tc_pattern = re.compile(r'\b(' + '|'.join(known_tables) + r')\.([A-Za-z_][A-Za-z0-9_]*)\b')

all_uncovered = set()
all_join_keys = set()
question_issues = {}

for q in questions:
    sql = q['SQL']
    refs = tc_pattern.findall(sql)
    issues = []
    for table, col in refs:
        concept, kind = find_covering_concept(table, col)
        if kind == 'uncovered_known_column':
            issues.append((table, col))
            all_uncovered.add((table, col))
        elif kind == 'join_key':
            all_join_keys.add((table, col))
    if issues:
        question_issues[q['question_id']] = issues

print(f"Join-key columns encountered across all questions (expected, not gaps): {len(all_join_keys)}")
for table, col in sorted(all_join_keys):
    print(f"  {table}.{col}")

print()
if question_issues:
    print(f"!! {len(question_issues)} / {len(questions)} questions reference a column with")
    print("   NO concept coverage (and is not a join key):\n")
    for qid, issues in question_issues.items():
        for table, col in issues:
            print(f"   [{qid}] {table}.{col}")
    print(f"\n   Distinct uncovered (table, column) pairs: {sorted(all_uncovered)}")
    print("\n   FIX: add a dimension or measure to financial_semantic_layer.yaml")
    print("   for each of these before running the pilot, or the LLM will be")
    print("   structurally unable to express these questions via concept-SQL.")
else:
    print(f"✓ All table.column references across all {len(questions)} questions are")
    print("  covered by an existing dimension, measure, or the join-key mechanism.")

# %% [markdown]
# ## 5. Same check for BARE (non-prefixed) columns in single-table
#     queries (many gold SQLs omit the table prefix when there's no
#     join, e.g. "WHERE status = 'D'" in a query touching only `loan`)

# %%
bare_issues = {}
for q in questions:
    sql = q['SQL']
    from_match = re.search(r'FROM\s+(\w+)', sql, re.IGNORECASE)
    has_join = bool(re.search(r'\bJOIN\b', sql, re.IGNORECASE))
    if not from_match or has_join:
        continue
    table = from_match.group(1)
    if table not in FINANCIAL_TABLES:
        continue
    for col in FINANCIAL_TABLES[table]:
        if re.search(r'\b' + re.escape(col) + r'\b', sql):
            concept, kind = find_covering_concept(table, col)
            if kind == 'uncovered_known_column':
                bare_issues.setdefault(q['question_id'], []).append((table, col))

if bare_issues:
    print(f"!! {len(bare_issues)} single-table questions use a bare column with no concept:")
    for qid, issues in bare_issues.items():
        for table, col in issues:
            print(f"   [{qid}] {table}.{col}")
else:
    print("✓ All bare (non-prefixed) column references in single-table queries")
    print("  are also covered by an existing concept.")

# %% [markdown]
# ## 6. MEASURE-MISUSE RISK ANALYSIS
#     Flags questions whose WHERE clause pattern resembles CR01's
#     failure mode: a scalar COUNT with no GROUP BY, filtering on a
#     condition that has a tempting-but-wrong measure available
#     (default_count/default_rate/running_count, female_count/
#     male_count, gold_count, owner_count).
#
#     This does NOT predict certain failure -- it identifies where
#     the temptation exists, so few-shot examples can be targeted
#     there. The compiler's WHERE-guard (added after the pilot) will
#     catch actual misuse as a clean compile_error rather than
#     invalid SQL, but avoiding the mistake in the first place still
#     saves API calls and improves effective accuracy.

# %%
MEASURE_TRIGGER_PATTERNS = {
    'default_count/rate/running_count risk': re.compile(
        r"status\s+(?:IN\s*\('[BD]'\s*,\s*'[BD]'\)|=\s*'[BD]')", re.IGNORECASE
    ),
    'female_count/male_count risk': re.compile(
        r"gender\s*=\s*'[MF]'", re.IGNORECASE
    ),
    'gold_count risk': re.compile(
        r"type\s*=\s*'gold'", re.IGNORECASE
    ),
    'owner_count risk': re.compile(
        r"type\s*=\s*'OWNER'", re.IGNORECASE
    ),
}

risk_report = []
for q in questions:
    sql = q['SQL']
    where_match = re.search(r'\bWHERE\b(.*?)(?:\bGROUP\s+BY\b|\bORDER\s+BY\b|$)', sql, re.IGNORECASE | re.DOTALL)
    where_section = where_match.group(1) if where_match else ''

    triggers = [label for label, pat in MEASURE_TRIGGER_PATTERNS.items() if pat.search(where_section)]
    if not triggers:
        continue

    has_group_by = bool(re.search(r'\bGROUP\s+BY\b', sql, re.IGNORECASE))
    is_scalar_count = bool(re.match(r'^\s*SELECT\s+COUNT\s*\(', sql, re.IGNORECASE)) and not has_group_by

    risk_report.append({
        'question_id': q['question_id'],
        'category': q['category'],
        'difficulty': q['difficulty'],
        'risk_level': 'HIGH' if is_scalar_count else 'MEDIUM',
        'triggers': triggers,
    })

high_risk = [r for r in risk_report if r['risk_level'] == 'HIGH']
medium_risk = [r for r in risk_report if r['risk_level'] == 'MEDIUM']

print(f"HIGH risk (scalar COUNT + no GROUP BY -- exactly CR01's shape): {len(high_risk)}")
for r in high_risk:
    print(f"  [{r['question_id']}] ({r['category']}, {r['difficulty']}): {r['triggers']}")

print(f"\nMEDIUM risk (trigger present, but has GROUP BY -- measure use in")
print(f"SELECT/HAVING would be CORRECT here, lower concern): {len(medium_risk)}")
for r in medium_risk:
    print(f"  [{r['question_id']}] ({r['category']}, {r['difficulty']}): {r['triggers']}")

print(f"\n{len(questions) - len(risk_report)} / {len(questions)} questions have no such trigger pattern.")

if high_risk:
    print("\n" + "=" * 70)
    print("RECOMMENDATION")
    print("=" * 70)
    print(f"""
{len(high_risk)} question(s) share CR01's exact risk shape. utils.py's
FEW_SHOT_EXAMPLES_V2 already includes one example targeting this
pattern (the "past due" / loan.status = 'D' example added after the
pilot). Verify that example is still present before running the
pilot -- if these HIGH-risk question_ids differ substantially from
what the existing few-shot covers (e.g. involve gender/card-type
filters rather than status), consider adding a second targeted
example for that specific trigger pattern.
""")
else:
    print("\n✓ No HIGH-risk questions found -- the existing few-shot coverage")
    print("  should be sufficient for this pattern.")

# %% [markdown]
# ## 7. Category / difficulty distribution sanity check
#     (confirms this question set matches the manuscript's stated
#     Phase 3 composition: 60 questions, 6 categories, 3 difficulty
#     levels with specific counts per Table in Section 3.3.3)

# %%
print("Category distribution:")
cat_counts = Counter(q['category'] for q in questions)
for cat, n in sorted(cat_counts.items()):
    print(f"  {cat}: {n}")

print(f"\nDifficulty distribution:")
diff_counts = Counter(q['difficulty'] for q in questions)
for diff in ['simple', 'moderate', 'challenging']:
    print(f"  {diff}: {diff_counts.get(diff, 0)}")

expected_categories = {
    'default_risk': 10, 'regional_risk': 11, 'transaction_behavior': 11,
    'loan_portfolio': 11, 'client_profile': 10, 'card_risk': 7,
}
expected_difficulty = {'simple': 22, 'moderate': 21, 'challenging': 17}

cat_mismatch = {k: (cat_counts.get(k, 0), v) for k, v in expected_categories.items() if cat_counts.get(k, 0) != v}
diff_mismatch = {k: (diff_counts.get(k, 0), v) for k, v in expected_difficulty.items() if diff_counts.get(k, 0) != v}

if not cat_mismatch and not diff_mismatch:
    print("\n✓ Category and difficulty distributions match the manuscript's")
    print("  Section 3.3.3 description exactly.")
else:
    if cat_mismatch:
        print(f"\n! Category count mismatches (found, expected): {cat_mismatch}")
    if diff_mismatch:
        print(f"! Difficulty count mismatches (found, expected): {diff_mismatch}")

# %% [markdown]
# ## Summary
# If sections 4-5 show no uncovered columns and section 7 confirms the
# expected distribution, the YAML's concept vocabulary is verified
# sufficient to express all 60 gold queries, and known risk patterns
# (section 6) are identified for targeted few-shot coverage. This
# clears the way to proceed to the pilot run without spending API
# budget discovering vocabulary gaps interactively.

SCRIPT VERSION: 2026-07-15-v1
Loaded 60 questions from ./credit_rating_questions_all.json
Loaded 7 cubes from ./financial_semantic_layer.yaml

Fields present in each question record: ['SQL', 'category', 'credit_rating_context', 'difficulty', 'evidence', 'question', 'question_id']
✓ All expected fields present.
Known tables: ['account', 'card', 'client', 'disp', 'district', 'loan', 'trans']

Simple dimension raw-column mappings per table:
  account: {'account_id': 'account_id', 'frequency': 'frequency', 'date': 'open_date'}
  district: {'district_id': 'district_id', 'A2': 'district_name', 'A3': 'region', 'A4': 'population', 'A11': 'avg_salary', 'A12': 'unemployment_rate_1995', 'A13': 'unemployment_rate_1996', 'A15': 'crimes_1995', 'A16': 'crimes_1996'}
  client: {'client_id': 'client_id', 'gender': 'gender', 'birth_date': 'birth_date'}
  loan: {'loan_id': 'loan_id', 'account_id': 'account_id', 'date': 'loan_date', 'duration': 'duration', 'payments': 'monthly_payment', 'status': 'status'